In [4]:
# %pip install datasets

     |████████████████████████████████| 512 kB 1.6 MB/s eta 0:00:01
     |████████████████████████████████| 110 kB 22.7 MB/s eta 0:00:01
     |████████████████████████████████| 24.7 MB 13.0 MB/s eta 0:00:01
     |████████████████████████████████| 268 kB 12.4 MB/s eta 0:00:01
     |████████████████████████████████| 116 kB 22.2 MB/s eta 0:00:01
     |████████████████████████████████| 53 kB 5.6 MB/s  eta 0:00:01
     |████████████████████████████████| 363 kB 20.1 MB/s eta 0:00:01
     |████████████████████████████████| 85 kB 7.0 MB/s eta 0:00:011
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
    Preparing wheel metadata ... done
     |████████████████████████████████| 143 kB 28.6 MB/s eta 0:00:01
     |████████████████████████████████| 189 kB 11.6 MB/s eta 0:00:01
     |████████████████████████████████| 62 kB 2.0 MB/s  eta 0:00:01
     |████████████████████████████████| 61 kB 1.0 MB/s  eta 0:00:01
     |█

In [11]:
import pandas as pd
from pathlib import Path
from datasets import load_dataset

# ---------- paths ----------
data_dir = Path("data")

def pick_path(stem: str) -> Path:
    gz = data_dir / f"{stem}.tsv.gz"
    tsv = data_dir / f"{stem}.tsv"
    if gz.exists():
        return gz
    if tsv.exists():
        return tsv
    raise FileNotFoundError(f"Missing both {gz} and {tsv}")

title_basics_path = pick_path("title.basics")
title_crew_path = pick_path("title.crew")
name_basics_path = pick_path("name.basics")

# ---------- 1) HuggingFace ----------
hf = load_dataset("fgiobergia/imdb-id", split="train").to_pandas()

# Normalize columns from HF
if "tconst" not in hf.columns and "movie_id" in hf.columns:
    hf = hf.rename(columns={"movie_id": "tconst"})
if "review" not in hf.columns and "text" in hf.columns:
    hf = hf.rename(columns={"text": "review"})

hf["tconst"] = hf["tconst"].astype(str).str.strip()
hf.to_csv("data/imdb_reviews.csv", index=False)

# ---------- 2) title.basics join ----------
basics = pd.read_csv(
    title_basics_path,
    sep="\t",
    dtype=str,
    na_values="\\N",
    usecols=["tconst", "primaryTitle", "startYear", "runtimeMinutes", "genres"],
)

base = hf.merge(basics, on="tconst", how="left")
movie_ids = set(base["tconst"].dropna())

# ---------- 3) title.crew (filter by tconst) ----------
crew_parts = []
for chunk in pd.read_csv(
    title_crew_path,
    sep="\t",
    dtype=str,
    na_values="\\N",
    usecols=["tconst", "directors"],
    chunksize=1_000_000,
):
    m = chunk["tconst"].isin(movie_ids)
    if m.any():
        crew_parts.append(chunk.loc[m, ["tconst", "directors"]])

crew = pd.concat(crew_parts, ignore_index=True) if crew_parts else pd.DataFrame(columns=["tconst", "directors"])

# ---------- 4) name.basics (filter by nconst) ----------
director_ids = (
    crew["directors"]
    .dropna()
    .str.split(",")
    .explode()
    .astype(str)
    .str.strip()
)
director_ids = set(director_ids[director_ids.ne("")])

name_parts = []
for chunk in pd.read_csv(
    name_basics_path,
    sep="\t",
    dtype=str,
    na_values="\\N",
    usecols=["nconst", "primaryName"],
    chunksize=1_000_000,
):
    m = chunk["nconst"].isin(director_ids)
    if m.any():
        name_parts.append(chunk.loc[m, ["nconst", "primaryName"]])

names = pd.concat(name_parts, ignore_index=True) if name_parts else pd.DataFrame(columns=["nconst", "primaryName"])
name_map = dict(zip(names["nconst"], names["primaryName"]))

def resolve_directors(val):
    if pd.isna(val):
        return pd.NA
    ids = [x.strip() for x in str(val).split(",") if x and x.strip()]
    resolved = [name_map.get(x) for x in ids if name_map.get(x)]
    if not resolved:
        return pd.NA
    # deduplicate, keep order
    return ", ".join(dict.fromkeys(resolved))

crew["director"] = crew["directors"].map(resolve_directors)

# ---------- 5) final ----------
final = base.merge(crew[["tconst", "director"]], on="tconst", how="left")

out = final.rename(
    columns={
        "tconst": "movie_id",
        "primaryTitle": "title",
        "startYear": "year",
        "runtimeMinutes": "runtime",
    }
)[["movie_id", "title", "director", "year", "runtime", "genres", "review"]]

out_path = data_dir / "imdb_joined_out_with_directors.csv"
out.to_csv(out_path, index=False)

print("saved:", out_path)
print("shape:", out.shape)
print("non_na_director:", out["director"].notna().sum())

out.head()

Found cached dataset parquet (/Users/annremizova/.cache/huggingface/datasets/fgiobergia___parquet/fgiobergia--imdb-id-a45fc192cbad9b4d/0.0.0/14a00e99c0d15a23649d0db8944380ac81082d4b021f398733dd84f3a6c569a7)


saved: data/imdb_joined_out_with_directors.csv
shape: (25000, 7)
non_na_director: 23722


,movie_id,title,director,year,runtime,genres,review
0,tt0114057,Othello,Oliver Parker,1995,123,"Drama,Romance",Working with one of the best Shakespeare sourc...
1,tt0334541,Tremors 4: The Legend Begins,S.S. Wilson,2004,101,"Action,Adventure,Comedy","Well...tremors I, the original started off in ..."
2,tt0337640,Hollywood North,Peter O'Brian,2003,89,Comedy,Ouch! This one was a bit painful to sit throug...
3,tt0219400,Waking Up in Reno,Jordan Brady,2002,91,"Comedy,Romance","I've seen some crappy movies in my life, but t..."
4,tt0806203,Carriers,"David Pastor, Àlex Pastor",2009,84,"Adventure,Drama,Horror","""Carriers"" follows the exploits of two guys an..."


In [12]:
# Save to CSV
out.to_csv("data/imdb_joined.csv", index=False)